In [0]:
# Inicialización de parámetros
dbutils.widgets.text("workspace", "workspace", "workspace")
dbutils.widgets.text("schema", "hiraoka", "schema")

# Extraer valores de los parámetros
workspace = dbutils.widgets.get("workspace")
schema = dbutils.widgets.get("schema")

In [0]:
# Leer tabla bronze
table_bronze = f"{workspace}.{schema}.bronze_celulares"
df_celulares = spark.read.table(table_bronze)

In [0]:
from pyspark.sql.functions import regexp_replace, trim, col, round

In [0]:
from pyspark.sql.functions import expr

# Limpieza de Marca: eliminar saltos de línea y espacios
df_silver = df_celulares.withColumn("Marca", trim(regexp_replace(col("Marca"), "\\n", "")))

# Limpieza de PrecioRegular: eliminar todo excepto dígitos y punto decimal
df_silver = df_silver.withColumn(
    "PrecioRegular",
    trim(regexp_replace(col("PrecioRegular"), "[^0-9.]", ""))
)
# Convertir a float con try_cast (SQL) - retorna NULL si no puede convertir
df_silver = df_silver.withColumn(
    "PrecioRegular",
    expr("try_cast(PrecioRegular as decimal(10,2))")
)

# Limpieza de PrecioOnline: eliminar todo excepto dígitos y punto decimal
df_silver = df_silver.withColumn(
    "PrecioOnline",
    trim(regexp_replace(col("PrecioOnline"), "[^0-9.]", ""))
)
# Convertir a float con try_cast (SQL) - retorna NULL si no puede convertir
df_silver = df_silver.withColumn(
    "PrecioOnline",
    expr("try_cast(PrecioOnline as decimal(10, 2))")
)

In [0]:
# Escribir el DataFrame transformado como tabla Silver
table_silver = f"{workspace}.{schema}.silver_celulares"

df_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_silver)